# 🎙️ YouTube → Transcripción + SRT

Pegá una o varias URLs de YouTube. El notebook descarga el audio, lo transcribe con **Whisper Large-v3 Turbo** y prepara un ZIP con los archivos `.txt` y `.srt`.

> **Antes de empezar:** en Colab elegí `Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU` (o cualquier GPU). Usá únicamente contenido propio o para el que tengas autorización.

In [ ]:
# Instalación (se ejecuta una sola vez por sesión)
!pip -q install -U transformers accelerate gradio yt-dlp sentencepiece
!apt-get -qq update && apt-get -qq install -y ffmpeg


In [ ]:
import os
import re
import shutil
import time
from pathlib import Path

import gradio as gr
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

WORKDIR = Path('/content/whisper_resultados')
AUDIO_DIR = WORKDIR / 'audio'
OUT_DIR = WORKDIR / 'transcriptos'
MODEL_ID = 'openai/whisper-large-v3-turbo'
asr = None

def cargar_modelo():
    global asr
    if asr is not None:
        return
    if not torch.cuda.is_available():
        raise gr.Error('No se detectó GPU. En Colab activá una GPU y volvé a ejecutar esta celda.')
    dtype = torch.float16
    model = AutoModelForSpeechSeq2Seq.from_pretrained(
        MODEL_ID, torch_dtype=dtype, low_cpu_mem_usage=True, use_safetensors=True
    ).to('cuda')
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    asr = pipeline(
        'automatic-speech-recognition', model=model, tokenizer=processor.tokenizer,
        feature_extractor=processor.feature_extractor, torch_dtype=dtype, device=0,
        chunk_length_s=30
    )

def seguro(nombre):
    nombre = re.sub(r'[^\w .-]+', '_', nombre, flags=re.UNICODE).strip()
    return nombre[:100] or 'video'

def tiempo_srt(segundos):
    ms = round((segundos - int(segundos)) * 1000)
    h, resto = divmod(int(segundos), 3600)
    m, s = divmod(resto, 60)
    return f'{h:02}:{m:02}:{s:02},{ms:03}'

def escribir_srt(chunks, destino):
    lineas = []
    for i, chunk in enumerate(chunks, 1):
        inicio, fin = chunk['timestamp']
        if inicio is None: continue
        if fin is None: fin = inicio + 2
        lineas += [str(i), f'{tiempo_srt(inicio)} --> {tiempo_srt(fin)}', chunk['text'].strip(), '']
    destino.write_text('\n'.join(lineas), encoding='utf-8')

def procesar(urls, idioma, progreso=gr.Progress()):
    lista = [u.strip() for u in urls.splitlines() if u.strip() and not u.strip().startswith('#')]
    if not lista:
        raise gr.Error('Pegá al menos una URL, una por línea.')
    progreso(0, desc='Preparando Whisper Large-v3 Turbo…')
    cargar_modelo()
    if WORKDIR.exists(): shutil.rmtree(WORKDIR)
    AUDIO_DIR.mkdir(parents=True)
    OUT_DIR.mkdir()
    import yt_dlp
    ydl_opts = {
        'format': 'bestaudio/best', 'outtmpl': str(AUDIO_DIR / '%(id)s.%(ext)s'),
        'quiet': True, 'noplaylist': True, 'postprocessors': []
    }
    hechos, errores = [], []
    for n, url in enumerate(lista, 1):
        try:
            progreso((n-1)/len(lista), desc=f'Video {n}/{len(lista)}: descargando audio…')
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                info = ydl.extract_info(url, download=True)
                audio = Path(ydl.prepare_filename(info))
            titulo = seguro(info.get('title', f'video_{n}'))
            progreso((n-.45)/len(lista), desc=f'Video {n}/{len(lista)}: transcribiendo…')
            kwargs = {'return_timestamps': True, 'generate_kwargs': {'task': 'transcribe'}}
            if idioma != 'Detectar automáticamente': kwargs['generate_kwargs']['language'] = idioma
            resultado = asr(str(audio), **kwargs)
            txt = OUT_DIR / f'{titulo}.txt'
            srt = OUT_DIR / f'{titulo}.srt'
            txt.write_text(resultado['text'].strip(), encoding='utf-8')
            escribir_srt(resultado.get('chunks', []), srt)
            hechos.append(titulo)
        except Exception as e:
            errores.append(f'• URL {n}: {str(e)[:180]}')
    zip_path = Path('/content/transcriptos_whisper.zip')
    if zip_path.exists(): zip_path.unlink()
    shutil.make_archive(str(zip_path.with_suffix('')), 'zip', OUT_DIR)
    progreso(1, desc='¡Terminado!')
    estado = f'### ✅ Listo: {len(hechos)} de {len(lista)} video(s) procesado(s)'
    if hechos: estado += '\n\n**Incluidos:** ' + ', '.join(hechos)
    if errores: estado += '\n\n### ⚠️ Algunos enlaces no se pudieron procesar\n' + '\n'.join(errores)
    return estado, str(zip_path)

CSS = '''
.gradio-container {max-width: 920px !important; background: #f8fafc;}
#hero {text-align:center; padding: 18px 8px 6px;}
#hero h1 {font-size: 2.25rem; margin-bottom: 4px;}
.primary-btn {background: linear-gradient(135deg,#e11d48,#be123c)!important; border:none!important; font-size:1.1rem!important; width:100%!important;}
.side-help {background:white!important;border:1px solid #f1d5dc!important;border-radius:12px!important;padding:8px 14px!important;margin-top:10px!important;}
'''

with gr.Blocks(theme=gr.themes.Soft(primary_hue='rose'), css=CSS, title='YouTube a texto') as demo:
    gr.HTML('<div id="hero"><h1>🎙️ YouTube a texto</h1><p>Transcripciones y subtítulos con Whisper Large-v3 Turbo</p></div>')
    with gr.Row(equal_height=False):
        with gr.Column(scale=3, min_width=360):
            urls = gr.Textbox(label='URLs de YouTube', lines=9, placeholder='https://www.youtube.com/watch?v=...\nhttps://youtu.be/...', info='Una URL por línea. Podés pegar varias a la vez.')
        with gr.Column(scale=1, min_width=240):
            idioma = gr.Dropdown(['Detectar automáticamente','es','en','pt','fr','it','de'], value='es', label='Idioma', info='Elegí “detectar” si varían.')
            gr.Markdown('#### El ZIP incluye\n- `video.txt` — texto completo\n- `video.srt` — subtítulos con tiempos', elem_classes='side-help')
    boton = gr.Button('✨ Procesar transcripciones', variant='primary', elem_classes='primary-btn')
    estado = gr.Markdown()
    descarga = gr.File(label='Descargá tus resultados', file_types=['.zip'], interactive=False)
    boton.click(procesar, inputs=[urls, idioma], outputs=[estado, descarga])

# La interfaz final está en la celda siguiente.


In [ ]:
# Interfaz mejorada: URLs individuales, vista previa de títulos y estado en vivo
MAX_URLS = 8

def duracion_legible(segundos):
    segundos = int(segundos or 0)
    minutos, segundos = divmod(segundos, 60)
    return f'{minutos} min {segundos:02} s' if minutos else f'{segundos} s'

def titulo_video(url):
    if not url or not url.strip():
        return 'ℹ️ Pegá una URL para ver el título del video.'
    try:
        import yt_dlp
        with yt_dlp.YoutubeDL({'quiet': True, 'noplaylist': True}) as ydl:
            info = ydl.extract_info(url.strip(), download=False)
        titulo = info.get('title', 'Video sin título')
        canal = info.get('channel') or info.get('uploader') or ''
        return f'🎬 **{titulo}**  \n{canal} · {duracion_legible(info.get("duration"))}'
    except Exception as e:
        return f'⚠️ No pude leer ese enlace: {str(e)[:140]}'

def revelar_campo(cantidad):
    siguiente = min(cantidad + 1, MAX_URLS)
    filas = [gr.update(visible=i < siguiente) for i in range(MAX_URLS)]
    return [siguiente, *filas, gr.update(visible=siguiente < MAX_URLS)]

def panel_estado(items):
    lineas = ['## Progreso de transcripción']
    for icono, texto, porcentaje in items:
        lineas.append(f'{icono} **{texto}**  \n<div class="video-progress"><i style="width:{porcentaje}%"></i></div>')
    return '\n\n'.join(lineas)

def procesar_campos(*entradas, progreso=gr.Progress()):
    idioma = entradas[-1]
    lista = [str(url).strip() for url in entradas[:-1] if str(url).strip()]
    if not lista:
        raise gr.Error('Pegá al menos una URL de YouTube.')
    estados = [('⏳', f'Video {i}: en espera', 0) for i in range(1, len(lista)+1)]
    yield panel_estado(estados), gr.update(value=None, visible=False)
    progreso(0, desc='Preparando Whisper Large-v3 Turbo…')
    cargar_modelo()
    if WORKDIR.exists(): shutil.rmtree(WORKDIR)
    AUDIO_DIR.mkdir(parents=True)
    OUT_DIR.mkdir()
    import yt_dlp
    hechos, errores = [], []
    for n, url in enumerate(lista, 1):
        try:
            estados[n-1] = ('🔽', f'Video {n}: descargando audio…', 18)
            progreso((n-1)/len(lista), desc=f'Video {n}/{len(lista)}: descargando audio…')
            yield panel_estado(estados), gr.update(value=None, visible=False)
            opciones = {'format': 'bestaudio/best', 'outtmpl': str(AUDIO_DIR / '%(id)s.%(ext)s'), 'quiet': True, 'noplaylist': True}
            with yt_dlp.YoutubeDL(opciones) as ydl:
                info = ydl.extract_info(url, download=True)
                audio = Path(ydl.prepare_filename(info))
            titulo = seguro(info.get('title', f'video_{n}'))
            # Estimación prudente para GPU T4: ~10% de la duración del audio + preparación.
            estimado = max(20, int((info.get('duration') or 300) * 0.10) + 12)
            estados[n-1] = ('🎙️', f'{titulo}: transcribiendo · estimado ≈ {duracion_legible(estimado)}', 48)
            progreso((n-.45)/len(lista), desc=f'Video {n}/{len(lista)}: transcribiendo…')
            yield panel_estado(estados), gr.update(value=None, visible=False)
            kwargs = {'return_timestamps': True, 'generate_kwargs': {'task': 'transcribe'}}
            if idioma != 'Detectar automáticamente': kwargs['generate_kwargs']['language'] = idioma
            resultado = asr(str(audio), **kwargs)
            (OUT_DIR / f'{titulo}.txt').write_text(resultado['text'].strip(), encoding='utf-8')
            escribir_srt(resultado.get('chunks', []), OUT_DIR / f'{titulo}.srt')
            hechos.append(titulo)
            estados[n-1] = ('✅', f'{titulo}: listo', 100)
        except Exception as e:
            errores.append(f'Video {n}: {str(e)[:150]}')
            estados[n-1] = ('⚠️', f'Video {n}: no se pudo procesar', 100)
        progreso(n/len(lista), desc=f'Video {n}/{len(lista)} completado')
        yield panel_estado(estados), gr.update(value=None, visible=False)
    if not hechos:
        raise gr.Error('No se pudo transcribir ningún video. Revisá los enlaces.')
    zip_path = Path('/content/transcriptos_whisper.zip')
    if zip_path.exists(): zip_path.unlink()
    shutil.make_archive(str(zip_path.with_suffix('')), 'zip', OUT_DIR)
    final = panel_estado(estados) + f'\n\n### ✅ Listo para descargar: {len(hechos)} video(s)'
    if errores: final += '\n\n⚠️ ' + '\n'.join(errores)
    yield final, gr.update(value=str(zip_path), visible=True)

CSS_MEJORADO = '''
.gradio-container {max-width: 980px !important; background:#f8fafc;}
#hero {text-align:center;padding:18px 8px 6px;} #hero h1 {font-size:2.25rem;margin-bottom:4px;}
.primary-btn {background:linear-gradient(135deg,#e11d48,#be123c)!important;border:none!important;font-size:1.1rem!important;width:100%!important;}
.url-card {border:1.5px solid #e2b4c0!important;border-radius:14px!important;background:white!important;padding:12px 14px!important;margin:10px 0!important;box-shadow:0 3px 12px #8813370d;}
.video-progress {height:8px;background:#fde7ed;border-radius:99px;overflow:hidden;margin-top:5px}.video-progress i {display:block;height:100%;background:linear-gradient(90deg,#fb7185,#e11d48);border-radius:99px;transition:width .5s}
.actions-row {align-items:end!important;background:white!important;border:1px solid #f1d5dc!important;border-radius:14px!important;padding:14px!important;margin-top:14px!important;} @media(max-width:640px){.actions-row{flex-wrap:wrap!important}.actions-row>*{min-width:100%!important}}
'''

with gr.Blocks(theme=gr.themes.Soft(primary_hue='rose'), css=CSS_MEJORADO, title='YouTube a texto') as demo_mejorada:
    gr.HTML('<div id="hero"><h1>🎙️ YouTube a texto</h1><p>Transcripciones y subtítulos con Whisper Large-v3 Turbo</p></div>')
    cantidad = gr.State(1)
    campos, filas = [], []
    for i in range(MAX_URLS):
        with gr.Group(visible=(i == 0), elem_classes='url-card') as fila:
            with gr.Row():
                campo = gr.Textbox(label=f'URL del video {i+1}', placeholder='https://www.youtube.com/watch?v=…', scale=5)
                ver_titulo = gr.Button('Ver título', scale=1, min_width=110)
            vista_titulo = gr.Markdown()
            ver_titulo.click(titulo_video, inputs=campo, outputs=vista_titulo)
        campos.append(campo); filas.append(fila)
    agregar = gr.Button('＋ Agregar otra URL', variant='secondary')
    agregar.click(revelar_campo, inputs=cantidad, outputs=[cantidad, *filas, agregar])
    with gr.Row(equal_height=False, elem_classes='actions-row'):
        idioma = gr.Dropdown(['Detectar automáticamente','es','en','pt','fr','it','de'], value='es', label='Idioma', scale=1, min_width=220)
        boton = gr.Button('✨ Procesar transcripciones', variant='primary', elem_classes='primary-btn', scale=2, min_width=280)
    estado = gr.Markdown()
    descarga = gr.File(label='✅ Tu archivo está listo', file_types=['.zip'], visible=False, interactive=False)
    boton.click(procesar_campos, inputs=[*campos, idioma], outputs=[estado, descarga])

demo_mejorada.launch(share=False, debug=True)
